In [ ]:
# Module B — Fine-Tuning Regimes (Internalising Multilingual Reasoning)
# -------------------------------------------------------------------------
# This notebook demonstrates a lightweight, reproducible example of
# multilingual fine-tuning and structure-aware evaluation.
# using a real dataset: juletxara/mgsm (Multilingual GSM8K).
# -------------------------------------------------------------------------

In [ ]:
FULL_MODE = True   # set False to run offline
MODEL_NAME = "utter-project/EuroLLM-1.7B-Instruct"
DATASET_NAME = "juletxara/mgsm"

LANGS = ["en", "it", "es", "zh"]
MAX_SAMPLES = 20   # keep it tiny for demonstration

import torch, re, pandas as pd
from datasets import load_dataset
import random, matplotlib.pyplot as plt


In [ ]:
# -------------------------------------------------------------
# 1 · Load Model (EuroLLM or fallback)
# -------------------------------------------------------------
try:
    if FULL_MODE:
        from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
        tok = AutoTokenizer.from_pretrained(MODEL_NAME)
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto"
        )
        gen = pipeline("text-generation", model=mdl, tokenizer=tok)
        print(f"✅ Loaded {MODEL_NAME}")
    else:
        gen = None
        print("⚙️ Using symbolic fallback (no model).")
except Exception as e:
    gen = None
    print("Fallback mode due to:", e)


In [ ]:
# -------------------------------------------------------------
# 2 · Load MGSM Dataset (subset for four languages)
# -------------------------------------------------------------
try:
    ds_all = load_dataset(DATASET_NAME)
except Exception as e:
    print("Could not download dataset, switching to mock mode:", e)
    ds_all = None

samples = []
if ds_all:
    for L in LANGS:
        subset = ds_all[L]["test"].shuffle(seed=42).select(range(min(MAX_SAMPLES, len(ds_all[L]["test"]))))
        for ex in subset:
            samples.append({
                "lang": L,
                "question": ex["question"],
                "answer": str(ex["answer"]),
                "rationale": ex["answer"]
            })
else:
    # fallback tiny synthetic math problems
    import random
    for L in LANGS:
        for _ in range(5):
            x, y = random.randint(1,9), random.randint(1,9)
            samples.append({
                "lang": L,
                "question": f"What is {x}+{y}?",
                "answer": str(x+y),
                "rationale": f"Step 1: add {x} and {y}. Step 2: result = {x+y}."
            })

df = pd.DataFrame(samples)
print("✅ Dataset loaded:", len(df), "samples")
df.sample(4)


In [ ]:
# -------------------------------------------------------------
# 3 · Define Structure Validation and Scoring
# -------------------------------------------------------------
def is_well_formed(text: str) -> bool:
    """Checks for explicit reasoning steps."""
    return bool(re.search(r"(Step|Passo|Paso|步)\s*\d+", text))

def distillation_score(student: str, target_answer: str) -> float:
    """Combines structure validity and numeric overlap with gold."""
    struct = 1.0 if is_well_formed(student) else 0.0
    nums_s = set(re.findall(r"\d+", student))
    nums_t = set(re.findall(r"\d+", target_answer))
    overlap = len(nums_s & nums_t) / max(1, len(nums_t))
    return 0.5 * struct + 0.5 * overlap


In [ ]:
# -------------------------------------------------------------
# 4 · Generation Function
# -------------------------------------------------------------
def model_generate(question: str, lang: str) -> str:
    """Generates structured reasoning with EuroLLM or fallback."""
    if gen is None:
        # simple deterministic rule
        m = re.findall(r"(\d+)\s*\+\s*(\d+)", question)
        if m:
            x, y = map(int, m[0])
            s = x + y
        else:
            s = random.randint(1,10)
        templates = {
            "en": f"Step 1: add {x} and {y}. Step 2: result = {s}. answer: {s}",
            "it": f"Passo 1: somma {x} e {y}. Passo 2: risultato = {s}. answer: {s}",
            "es": f"Paso 1: suma {x} y {y}. Paso 2: resultado = {s}. answer: {s}",
            "zh": f"第1步：把 {x} 和 {y} 相加。第2步：结果 = {s}。答案：{s}"
        }
        return templates.get(lang, str(s))
    else:
        prompt = f"You are a helpful assistant. Respond in {lang}.\nExplain step by step:\n{question}"
        out = gen(prompt, max_new_tokens=128, do_sample=False)[0]["generated_text"]
        return out


In [ ]:
# -------------------------------------------------------------
# 5 · Evaluation Loop
# -------------------------------------------------------------
results = []
for _, row in df.iterrows():
    out = model_generate(row["question"], row["lang"])
    valid = is_well_formed(out)
    score = distillation_score(out, row["answer"])
    results.append({"lang": row["lang"], "valid": valid, "distill_score": score})

df_res = pd.DataFrame(results)
eval_summary = df_res.groupby("lang").agg(
    valid_rate=("valid","mean"),
    distill_score=("distill_score","mean")
).reset_index()
eval_summary


In [ ]:
# -------------------------------------------------------------
# 5bis · Simulated GRPO Step (Group Relative Preference Optimisation)
# -------------------------------------------------------------
# GRPO encourages the model to align with preferred reasoning traces by comparing multiple multilingual completions within a group and assigning relative rewards based on structure, coherence, and diversity.

def grpo_score(outputs):
    """Compute a relative preference score across multilingual group outputs."""
    # Normalised distillation scores per sample
    base = [distillation_score(o, re.findall(r"\d+", o)[-1] if re.findall(r"\d+", o) else "0") for o in outputs]
    mean_val = sum(base) / len(base)
    rel = [b - mean_val for b in base]
    return rel

def simulate_grpo_phase(df, n_groups=5):
    """Simulate one GRPO update phase with multilingual completions."""
    rewards = []
    for _ in range(n_groups):
        group = df.sample(len(LANGS))  # one item per language
        outs = [model_generate(q, L) for q, L in zip(group["question"], group["lang"])]
        rel_pref = grpo_score(outs)
        for (L, r) in zip(group["lang"], rel_pref):
            rewards.append({"lang": L, "reward": r})
    return pd.DataFrame(rewards).groupby("lang").mean().reset_index()

# Run simulated GRPO
grpo_results = simulate_grpo_phase(df)
print("Simulated GRPO group rewards:")
display(grpo_results)

# Integrate GRPO signal with evaluation results
merged = eval_summary.merge(grpo_results, on="lang", how="left")
merged["adjusted_score"] = merged["distill_score"] + 0.2 * merged["reward"]
display(merged)

In [ ]:
# -------------------------------------------------------------
# 5ter · Optional DPO Simulation (Direct Preference Optimisation)
# -------------------------------------------------------------
# DPO directly optimises the model by increasing log-probabilities of preferred responses and decreasing those of rejected ones.
# Here we simulate this process using our distillation_score as proxy for the 'preference' signal.

def simulate_dpo_pair(question, lang):
    """
    Generate a pair (preferred, rejected) for a given question/language.
    Preferred: baseline model output.
    Rejected: perturbed or structurally incomplete variant.
    """
    base = model_generate(question, lang)
    # Simulate a weaker answer by truncation or corruption
    if len(base) > 40:
        rej = base[:len(base)//2] + " ..."
    else:
        rej = base.replace("Step 2", "").replace("Passo 2", "").replace("Paso 2", "").replace("第2步", "")
    return base, rej

def compute_dpo_loss(preferred, rejected, gold_answer):
    """
    Simulate DPO loss as preference difference between two responses.
    Positive when preferred aligns better with structure/answer.
    """
    s_pref = distillation_score(preferred, gold_answer)
    s_rej = distillation_score(rejected, gold_answer)
    diff = s_pref - s_rej
    # Logistic DPO-like loss: -log(σ(β*(s_pref - s_rej)))
    import math
    beta = 2.0
    loss = -math.log(1 / (1 + math.exp(-beta * diff)))
    return diff, loss

# Run simulated DPO on small multilingual subset
dpo_records = []
for _, row in df.sample(8).iterrows():
    pref, rej = simulate_dpo_pair(row["question"], row["lang"])
    diff, loss = compute_dpo_loss(pref, rej, row["answer"])
    dpo_records.append({"lang": row["lang"], "delta_pref": diff, "dpo_loss": loss})

df_dpo = pd.DataFrame(dpo_records)
dpo_summary = df_dpo.groupby("lang").mean().reset_index()

print("Simulated DPO preference optimisation results:")
display(dpo_summary)

# Merge with evaluation summary to illustrate alignment effect
merged_dpo = eval_summary.merge(dpo_summary, on="lang", how="left")
merged_dpo["adjusted_score_dpo"] = merged_dpo["distill_score"] + 0.2 * merged_dpo["delta_pref"]
display(merged_dpo)


In [ ]:
# -------------------------------------------------------------
# 6 · Visualise
# -------------------------------------------------------------
plt.figure(figsize=(6,4))
plt.bar(eval_summary["lang"], eval_summary["distill_score"], color="teal")
plt.ylabel("Average Distillation Score")
plt.xlabel("Language")
plt.title("Structure Awareness across Languages (EuroLLM + MGSM)")
plt.grid(axis="y", alpha=0.3)
plt.show()